In [0]:
# ============================================================
# Retail Intelligence Brasil
# Notebook.......: 14_bronze_sidra_populacao
# Camada.........: Bronze
# Fonte..........: IBGE - SIDRA
# Tabela.........: 4709
# Indicador......: População residente
# Objetivo.......: Persistir dados da tabela 4709 na Bronze
# Autor..........: Cristhina Freire
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import requests
import json

from datetime import datetime

# ============================================================
# 2. CONFIGURAÇÃO
# ============================================================

CATALOG = "retail_intelligence"
SCHEMA = "bronze"

TABLE_NAME = "ibge_sidra_populacao_raw"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

BASE_URL = "https://apisidra.ibge.gov.br"

SIDRA_TABLE_ID = 4709
SIDRA_VARIABLE_ID = 93
SIDRA_PERIOD = "2022"
SIDRA_LEVEL = "n6"
SIDRA_LOCATION = "all"

URL = (
    f"{BASE_URL}/values/"
    f"t/{SIDRA_TABLE_ID}/"
    f"{SIDRA_LEVEL}/{SIDRA_LOCATION}/"
    f"v/{SIDRA_VARIABLE_ID}/"
    f"p/{SIDRA_PERIOD}"
)

TIMEOUT = 60

CURRENT_DATE = datetime.now().strftime("%Y-%m-%d")
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/sidra/demografia/populacao/{CURRENT_DATE}"
)

FILE_NAME = f"populacao_{CURRENT_TIME}.json"

FILE_PATH = f"{VOLUME_PATH}/{FILE_NAME}"

print("=" * 60)
print("BRONZE - SIDRA - POPULAÇÃO RESIDENTE")
print("=" * 60)

# ============================================================
# 3. EXTRAÇÃO
# ============================================================

print("Consumindo API SIDRA...")

response = requests.get(
    URL,
    timeout=TIMEOUT
)

response.raise_for_status()

sidra_response = response.json()

print(f"Registros retornados pela API: {len(sidra_response)}")

# ============================================================
# 4. REMOVER CABEÇALHO
# ============================================================

header = sidra_response[0]

landing_data = sidra_response[1:]

print("Cabeçalho identificado.")

print(f"Registros válidos: {len(landing_data)}")

# ============================================================
# 5. PERSISTÊNCIA DO JSON
# ============================================================

print("Salvando JSON na Landing...")

dbutils.fs.mkdirs(VOLUME_PATH)

dbutils.fs.put(
    FILE_PATH,
    json.dumps(
        landing_data,
        ensure_ascii=False,
        indent=2
    ),
    overwrite=True
)

print(f"Arquivo salvo em:\n{FILE_PATH}")

# ============================================================
# 6. LEITURA DO JSON
# ============================================================

print("Lendo JSON salvo...")

landing_df = (
    spark.read
         .option("multiline", "true")
         .json(FILE_PATH)
)

print("Leitura concluída.")

# ============================================================
# 7. PERSISTÊNCIA NA BRONZE
# ============================================================

print("Persistindo tabela Delta...")

(
    landing_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
)

print("Tabela criada com sucesso.")

# ============================================================
# 8. INSPEÇÃO
# ============================================================

bronze_df = spark.table(FULL_TABLE_NAME)

print("=" * 60)
print("INSPEÇÃO")
print("=" * 60)

print(f"Tabela...........: {FULL_TABLE_NAME}")
print(f"Total Registros..: {bronze_df.count()}")

bronze_df.printSchema()

display(bronze_df)

# ============================================================
# 9. ENCERRAMENTO
# ============================================================

print("=" * 60)
print("Pipeline finalizado com sucesso.")
print("=" * 60)

BRONZE - SIDRA - POPULAÇÃO RESIDENTE
Consumindo API SIDRA...
Registros retornados pela API: 5571
Cabeçalho identificado.
Registros válidos: 5570
Salvando JSON na Landing...
Wrote 1374299 bytes.
Arquivo salvo em:
/Volumes/retail_intelligence/bronze/landing/ibge/sidra/demografia/populacao/2026-07-14/populacao_20260714_191019.json
Lendo JSON salvo...
Leitura concluída.
Persistindo tabela Delta...
Tabela criada com sucesso.
INSPEÇÃO
Tabela...........: retail_intelligence.bronze.ibge_sidra_populacao_raw
Total Registros..: 5570
root
 |-- D1C: string (nullable = true)
 |-- D1N: string (nullable = true)
 |-- D2C: string (nullable = true)
 |-- D2N: string (nullable = true)
 |-- D3C: string (nullable = true)
 |-- D3N: string (nullable = true)
 |-- MC: string (nullable = true)
 |-- MN: string (nullable = true)
 |-- NC: string (nullable = true)
 |-- NN: string (nullable = true)
 |-- V: string (nullable = true)



D1C,D1N,D2C,D2N,D3C,D3N,MC,MN,NC,NN,V
1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,45,Pessoas,6,Município,21494
1100023,Ariquemes - RO,93,População residente,2022,2022,45,Pessoas,6,Município,96833
1100031,Cabixi - RO,93,População residente,2022,2022,45,Pessoas,6,Município,5351
1100049,Cacoal - RO,93,População residente,2022,2022,45,Pessoas,6,Município,86887
1100056,Cerejeiras - RO,93,População residente,2022,2022,45,Pessoas,6,Município,15890
1100064,Colorado do Oeste - RO,93,População residente,2022,2022,45,Pessoas,6,Município,15663
1100072,Corumbiara - RO,93,População residente,2022,2022,45,Pessoas,6,Município,7519
1100080,Costa Marques - RO,93,População residente,2022,2022,45,Pessoas,6,Município,12627
1100098,Espigão D'Oeste - RO,93,População residente,2022,2022,45,Pessoas,6,Município,29414
1100106,Guajará-Mirim - RO,93,População residente,2022,2022,45,Pessoas,6,Município,39387


Pipeline finalizado com sucesso.
